# VetAI PoC - stage-by-stage walkthrough

This notebook runs the PoC pipeline one stage at a time so you can see exactly what happens between a question and a cited answer.

```
retrieve  ->  synthesize  ->  VERIFY (deterministic gate)  ->  generate
```

The rule the whole thing exists to protect: **retrieve and verify before writing; the vet decides.** Nothing unsupported reaches the answer.

**Before running:** drop your two PDFs into `PoC/books/`, copy `.env.example` to `.env` with your `OPENAI_API_KEY`, then run `python -m src.embed_index` once from the `PoC` folder.

## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import config
from src import embed_index, generate, ingest, pipeline, retrieve, synthesize, verify

for key, value in config.summary().items():
    print(f"{key:18s} {value}")

In [ ]:
collection = embed_index.get_collection()
indexed = collection.count()
print(f"vectors in index: {indexed:,}")

if indexed == 0:
    print("\nIndex is empty. From the PoC folder run:")
    print("    python -m src.embed_index")

## Stage 0 - chunking

Each book is sliced into token windows. One chunk = one vector = the unit we retrieve and cite.

```
stride   = chunk_size - overlap
#chunks ~= total_tokens / stride
```

A token is about 3/4 of a word, so the default 600-token chunk is roughly one textbook page. Overlap repeats the tail of one chunk at the head of the next, so an idea split across a boundary isn't lost.

In [ ]:
chunks = ingest.load_chunks()
total_tokens = sum(c["token_count"] for c in chunks)

print(f"chunks          : {len(chunks):,}")
print(f"total tokens    : {total_tokens:,}")
print(f"chunk_size      : {config.CHUNK_SIZE}")
print(f"overlap         : {config.CHUNK_OVERLAP}")
print(f"stride          : {config.stride()}")
print(f"avg tokens/chunk: {total_tokens // max(1, len(chunks))}")

books = sorted({c["book_title"] for c in chunks})
print(f"\nbooks indexed   : {books}")

sample = chunks[len(chunks) // 2]
print(f"\nsample chunk")
print(f"  chunk_id : {sample['chunk_id']}")
print(f"  source   : {sample['book_title']}, p.{sample['page']}")
print(f"  tokens   : {sample['token_count']}")
print(f"  text     : {sample['text'][:300]}...")

## Stage 1 - retrieve

The question is embedded with the same model as the chunks, then we ask the vector store for the closest passages. Each one carries its provenance (`chunk_id`, book, page) so it can be cited later.

In [ ]:
QUESTION = "What are the common causes of chronic kidney disease in cats?"

passages = retrieve.retrieve(QUESTION)

for p in passages:
    print(f"{p['similarity']:.3f}  {p['chunk_id']:30s}  {p['book_title']}, p.{p['page']}")
    print(f"       {p['text'][:170]}...\n")

## Stage 2 - synthesize

The LLM sees **only** those passages and must return discrete factual claims, each tagged with the `chunk_id` it came from. It is not trusted at this point - the next stage checks its work.

In [ ]:
synth = synthesize.synthesize(QUESTION, passages)

print(f"answerable: {synth['answerable']}")
if synth.get("notes"):
    print(f"notes     : {synth['notes']}")
print()

for i, claim in enumerate(synth["claims"], 1):
    print(f"{i}. {claim['text']}")
    print(f"   sources: {claim['sources']}\n")

## Stage 3 - VERIFY (the deterministic gate)

This is the component that makes the product trustworthy, and it is **not** an LLM. For each claim it checks:

1. does it cite a `chunk_id` that was actually retrieved?
2. does that passage measurably support it?
   `score = w * lexical_coverage + (1 - w) * cosine_similarity`

Claims scoring below the threshold are **dropped with a reason** and never reach the answer. Same claim + same passage always gives the same verdict.

In [ ]:
verdict = verify.verify_claims(synth["claims"], passages)

print(f"threshold = {verdict['threshold']}\n")

print(f"VERIFIED ({len(verdict['verified'])})")
for claim in verdict["verified"]:
    print(f"  score={claim['score']:.3f}  lex={claim['lexical']:.2f}  cos={claim['cosine']:.2f}")
    print(f"    {claim['text'][:110]}")
    print(f"    best source: {claim['best_source']}\n")

print(f"REJECTED ({len(verdict['rejected'])})")
for claim in verdict["rejected"]:
    print(f"  {claim['reason']:22s} score={claim['score']:.3f}")
    print(f"    {claim['text'][:110]}\n")

## Stage 4 - generate

Only the surviving claims and their approved citations go to the LLM. If nothing survived, the pipeline returns the honest "no evidence" answer **without calling the LLM at all** - so there is no path by which an unsupported answer can be produced.

In [ ]:
written = generate.generate(QUESTION, verdict["verified"], passages)

print(written["answer"])
print("\n" + "-" * 60)
print("Sources")
for citation in written["citations"]:
    print(f"  [{citation['marker']}] {citation['book_title']}, p.{citation['page']}")
    print(f"      {citation['text'][:160]}...")

## The whole pipeline in one call

This is what the Streamlit app uses.

In [ ]:
result = pipeline.answer_question(QUESTION)
print(pipeline.format_result(result))

## The grounding test (the one that matters)

Ask something the books cannot answer. A normal chatbot would invent an answer. This pipeline should retrieve weak passages, fail to verify any claim, and **say so** instead.

In [ ]:
OFF_TOPIC = "What is the recommended tyre pressure for a Boeing 747?"

probe = pipeline.answer_question(OFF_TOPIC)

print(probe["answer"])
print()
print(f"grounded        : {probe['grounded']}")
print(f"claims proposed : {len(probe['claims'])}")
print(f"claims verified : {len(probe['verified'])}")
print(f"claims dropped  : {len(probe['rejected'])}")

## Tuning

Try changing these and re-running - they are the levers that matter:

| Lever | Effect |
|---|---|
| `top_k` | how much evidence the synthesiser sees |
| `threshold` | how strict the verify gate is (higher = fewer, safer claims) |
| `CHUNK_SIZE` / `CHUNK_OVERLAP` in `.env` | retrieval precision vs context (needs a re-index) |

A stricter threshold is the honest lever: it trades completeness for certainty.

In [ ]:
for t in (0.5, 0.65, 0.8):
    trial = pipeline.answer_question(QUESTION, threshold=t)
    print(f"threshold {t:.2f}  ->  verified {len(trial['verified']):2d} / dropped {len(trial['rejected']):2d}  (grounded={trial['grounded']})")